<a href="https://colab.research.google.com/github/MdShajalalsojib/Data-Mining-Lab/blob/main/Feature_Engineering_Technigues_002.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Feature Engineering Technigues**

In [1]:
%%capture
!pip install pandas numpy scikit-learn matplotlib

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## **libraries**

In [3]:

from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    OneHotEncoder,
    PolynomialFeatures
)

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.datasets import fetch_california_housing

pd.set_option('display.precision', 6)

**Load Dataset**

In [4]:
data = fetch_california_housing(as_frame=True)
df = data.frame.copy()
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


# **Data Cleaning**

**De-duplicate and Missing**

In [5]:
df = df.drop_duplicates()
df.isnull().sum()

,0
MedInc,0
HouseAge,0
AveRooms,0
AveBedrms,0
Population,0
AveOccup,0
Latitude,0
Longitude,0
MedHouseVal,0


**Binning demonstration**

In [6]:
toy = [4,8,15,16,23,42,54,55,57,60,62,64,68,70,72]
toy.sort()
bins = np.array_split(toy, 3)
for i,b in enumerate(bins,1):
 print(f"Bin {i}: {b} -> Mean: {np.mean(b):.2f}")

Bin 1: [ 4  8 15 16 23] -> Mean: 13.20
Bin 2: [42 54 55 57 60] -> Mean: 53.60
Bin 3: [62 64 68 70 72] -> Mean: 67.20


**Outlier Detection using IQR**

In [7]:
Q1 = df['Population'].quantile(0.25)

Q3 = df['Population'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR

upper = Q3 + 1.5 * IQR

outliers_mask = (
    (df['Population'] < lower) |
    (df['Population'] > upper)
)

outliers_mask.sum(), lower, upper

(np.int64(1196), np.float64(-620.0), np.float64(3132.0))

# **Data Transformation**

**Scaling**

In [8]:
# MinMax and Standard Scaling
mm = MinMaxScaler()
df['MedInc_minmax'] = mm.fit_transform(df[['MedInc']])
ss = StandardScaler()
df['AveBedrms_z'] = ss.fit_transform(df[['AveBedrms']])
df[['MedInc','MedInc_minmax','AveBedrms','AveBedrms_z']].head()

,MedInc,MedInc_minmax,AveBedrms,AveBedrms_z
0,8.3252,0.539668,1.023810,-0.153758
1,8.3014,0.538027,0.971880,-0.263336
2,7.2574,0.466028,1.073446,-0.049016
3,5.6431,0.354699,1.073059,-0.049833
4,3.8462,0.230776,1.081081,-0.032906


# **Discretization**

In [9]:
age_bins = [0, 10, 30, 52]

age_labels = ['New', 'Mid', 'Old']

df['HouseAgeBin'] = pd.cut(
    df['HouseAge'],
    bins=age_bins,
    labels=age_labels,
    right=True,
    include_lowest=True
)

df[['HouseAge', 'HouseAgeBin']].head(10)

,HouseAge,HouseAgeBin
0,41.0,Old
1,21.0,Mid
2,52.0,Old
3,52.0,Old
4,52.0,Old
5,52.0,Old
6,52.0,Old
7,52.0,Old
8,42.0,Old
9,52.0,Old


**One-Hot Encoding**

In [10]:
lon_median = df['Longitude'].median()

df['Region'] = np.where(
    df['Longitude'] <= lon_median,
    'West',
    'East'
)

cols_to_encode = ['HouseAgeBin', 'Region']

dummies = pd.get_dummies(
    df[cols_to_encode],
    dtype=int
)

df = pd.concat(
    [df, dummies],
    axis=1
)

df.filter(
    regex='^(HouseAgeBin|Region)_'
).head()

,HouseAgeBin_New,HouseAgeBin_Mid,HouseAgeBin_Old,Region_East,Region_West
0,0,0,1,0,1
1,0,1,0,0,1
2,0,0,1,0,1
3,0,0,1,0,1
4,0,0,1,0,1


**Feature Engineering**

In [11]:
df['Rooms_per_Person'] = (
    df['AveRooms'] / (df['AveOccup'] + 1e-5)
)

df['Income_per_Room'] = (
    df['MedInc'] / (df['AveRooms'] + 1e-5)
)

df['Log_Population'] = np.log1p(
    df['Population']
)

df[[
    'AveRooms',
    'AveOccup',
    'Rooms_per_Person',
    'MedInc',
    'Income_per_Room',
    'Population',
    'Log_Population'
]].head()

,AveRooms,AveOccup,Rooms_per_Person,MedInc,Income_per_Room,Population,Log_Population
0,6.984127,2.555556,2.732909,8.3252,1.192016,322.0,5.777652
1,6.238137,2.109842,2.956671,8.3014,1.330748,2401.0,7.784057
2,8.288136,2.802260,2.957651,7.2574,0.875636,496.0,6.208590
3,5.817352,2.547945,2.283145,5.6431,0.970045,558.0,6.326149
4,6.281853,2.181467,2.879633,3.8462,0.612271,565.0,6.338594


**Dimensionality Reduction (PCA)**

In [12]:
pca_inputs = [
    'MedInc',
    'AveRooms',
    'AveOccup',
    'Population'
]

scaler_pca = StandardScaler()

X_pca = scaler_pca.fit_transform(
    df[pca_inputs]
)

pca = PCA(
    n_components=2,
    random_state=0
)

pca_components = pca.fit_transform(
    X_pca
)

df['PCA1'] = pca_components[:, 0]
df['PCA2'] = pca_components[:, 1]

df[['PCA1', 'PCA2']].head()

,PCA1,PCA2
0,2.206040,-0.343773
1,1.720496,0.901644
2,2.168604,-0.318344
3,0.866147,-0.419269
4,0.344351,-0.594164


**Feature Selection and Evaluation**

In [13]:
feature_cols = [
    'MedInc',
    'AveRooms',
    'AveOccup',
    'Population',
    'MedInc_minmax',
    'AveBedrms_z',
    'Rooms_per_Person',
    'Income_per_Room',
    'Log_Population',
    'PCA1',
    'PCA2'
] + list(dummies.columns)


X = df[feature_cols].copy()

y = df['MedHouseVal'].copy()


scaler_lin = StandardScaler()

X_std = scaler_lin.fit_transform(X)


lin = LinearRegression()

lin.fit(X_std, y)


importance = pd.Series(
    lin.coef_,
    index=feature_cols
).sort_values(
    ascending=False
)


importance

,0
Rooms_per_Person,0.555959
MedInc,0.422600
MedInc_minmax,0.422600
AveBedrms_z,0.172262
Income_per_Room,0.164168
HouseAgeBin_Old,0.088036
Log_Population,0.051624
PCA2,0.025441
Region_East,0.018781
Region_West,-0.018781


# **Integrating Everything into a Full Pipeline**

**Dataset Loading**

In [14]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    StandardScaler
)
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# Titanic train dataset columns:
# Survived, Pclass, Name, Sex, Age, SibSp, Parch,
# Ticket, Fare, Cabin, Embarked

df = pd.read_csv(
    "https://raw.githubusercontent.com/Ataullha/CSE-436-Data-Mining-Lab/refs/heads/main/titanic_train.csv"
)

df.head()

,passenger_id,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest,survived
0,1216,3,"Smyth, Miss. Julia",female,NaN,0,0,335432,7.7333,NaN,Q,13,NaN,NaN,1
1,699,3,"Cacic, Mr. Luka",male,38.0,0,0,315089,8.6625,NaN,S,NaN,NaN,Croatia,0
2,1267,3,"Van Impe, Mrs. Jean Baptiste (Rosalie Paula Go...",female,30.0,1,1,345773,24.1500,NaN,S,NaN,NaN,NaN,0
3,449,2,"Hocking, Mrs. Elizabeth (Eliza Needs)",female,54.0,1,3,29105,23.0000,NaN,S,4,NaN,"Cornwall / Akron, OH",1
4,576,2,"Veal, Mr. James",male,40.0,0,0,28221,13.0000,NaN,S,NaN,NaN,"Barre, Co Washington, VT",0


**Train/Test Split**

In [15]:
X = df.drop(columns=["survived"])

y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

**Simple Custom Feature Function**

In [16]:
def add_basic_features(df):
    """Create simple, easy-to-understand new features."""

    data = df.copy()

    # Total family members (self + siblings/spouse + parents/children)
    data["FamilySize"] = (
        data["sibsp"] + data["parch"] + 1
    )

    # Whether the passenger is alone
    data["IsAlone"] = (
        data["FamilySize"] == 1
    ).astype(int)

    # Extract short title from passenger's name
    data["Title"] = data["name"].str.extract(
        " ([A-Za-z]+)\.",
        expand=False
    )

    return data.drop(
        columns=["name"]
    )

<>:18: SyntaxWarning: invalid escape sequence '\.'
<>:18: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_2950/1384062476.py:18: SyntaxWarning: invalid escape sequence '\.'
  " ([A-Za-z]+)\.",


**Preprocessing Stage**

In [17]:
num_cols = [
    "age",
    "sibsp",
    "parch",
    "fare",
    "FamilySize",
    "IsAlone"
]

cat_cols = [
    "sex",
    "embarked",
    "pclass",
    "Title"
]


preprocess = ColumnTransformer([
    (
        "num",
        Pipeline([
            (
                "impute",
                SimpleImputer(strategy="median")
            ),
            (
                "scale",
                StandardScaler()
            )
        ]),
        num_cols
    ),
    (
        "cat",
        OneHotEncoder(
            handle_unknown="ignore"
        ),
        cat_cols
    )
])

**Pipeline Assembly and Model Training**

In [20]:
pipeline = Pipeline([
    ("features", feature_step),
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Full Feature Engineering Pipeline")
print(f"Accuracy: {accuracy:.4f}")

NameError: name 'feature_step' is not defined